In [3]:
#Install the relevent packages into the environment
!python -m pip install pennylane --upgrade
!python -m pip install numpy
!python -m pip install matplotlib
!python -m pip install pandas
!python -m pip install "jax<0.11"
!python -m pip install optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 87.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 40.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 56.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 84.0 MB/s  0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pennylane]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 44.2 MB/s  0:00:01
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.11.1
    Uninstalling jaxlib-0.11.1:
      Successfully uninstalled jaxlib-0.11.1
  Attempting uninstall: jax
    Found existing installation: jax 0.11.1
    Uninstalling jax-0.11.1

https://pennylane.ai/demos/tutorial_qnn_multivariate_regression

In [4]:
# Import required Packages partially from Reference [2]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qp
from pennylane import numpy as pnp
import jax
import io
from jax import numpy as jnp
import optax
# Set jax to use 8 byte precision
jax.config.update("jax_enable_x64", True)


########################## Changeable Parameters ###############################
# Set number of qubits/wires
N = 2
wires = 2 # Set number of wires
layers = 9 # Set number of layers
Perc_Train = 0.3

# Initialize quantum circuit
dev = qp.device('default.qubit', wires=N)

In [5]:
# Import package to load a file in google colab
from google.colab import files
uploaded = files.upload() # Request a file to be uploaded "Surface_features.csv"

### After running this section. press choose file and select Surface_features.csv ###

Saving Surface_features.csv to Surface_features.csv


In [6]:
# from Reference [2]
# Define a function for data embedding layers
def SZ(x):
   qp.AngleEmbedding(x, wires=range(N),rotation='Z')
def SX(x):
   qp.AngleEmbedding(x, wires=range(N),rotation='X')
def SY(x):
   qp.AngleEmbedding(x, wires=range(N),rotation='Y')
# Define a function for trainable layers
def W(params):
    qp.StronglyEntanglingLayers(params, wires=range(N))

In [7]:
# Import the primary input features from the csv file from Ref [1]
Geometry = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0, usecols=(4,6,7,8,9,10,11,12)).iloc[0:45].values

# Import the output features and its normailzation factor
Ra_ch = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(5,)).iloc[0:45].values
Ra_ch = Ra_ch[:,0]
ks = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(13,)).iloc[0:45].values
ks = ks[:,0]

# Data reconfiguration, to be used in ML from Reference [1]

X = Geometry
y = np.divide(ks,Ra_ch)


X[:,0] = np.divide(X[:,0],Ra_ch)
X[:,2] = np.abs(X[:,2])


# Generate secondary features and append them to the original dataset

n,m = X.shape
X0 = np.ones((n,1))
X1 = np.ones((n,1))
X2 = np.ones((n,1))
X3 = np.ones((n,1))
X4 = np.ones((n,1))
X5 = np.ones((n,1))
X6 = np.ones((n,1))
X7 = np.ones((n,1))
X8 = np.ones((n,1))
X9 = np.ones((n,1))
X10 = np.zeros((n,1))



X1[:,0] = np.transpose(X[:,4]*X[:,5])
X2[:,0] = np.transpose(X[:,4]*X[:,6])
X3[:,0] = np.transpose(X[:,4]*X[:,7])
X4[:,0] = np.transpose(X[:,5]*X[:,6])
X5[:,0] = np.transpose(X[:,5]*X[:,7])
X6[:,0] = np.transpose(X[:,6]*X[:,7])
X7[:,0] = np.transpose(X[:,4]*X[:,4])
X8[:,0] = np.transpose(X[:,5]*X[:,5])
X9[:,0] = np.transpose(X[:,6]*X[:,6])




X = np.hstack((X,X1))
X = np.hstack((X,X2))
X = np.hstack((X,X3))
X = np.hstack((X,X4))
X = np.hstack((X,X5))
X = np.hstack((X,X6))
X = np.hstack((X,X7))
X = np.hstack((X,X8))
X = np.hstack((X,X9))
X = np.hstack((X,X10)) # Padding array of zeros
X = np.hstack((X,X10)) # Padding array of zeros
X = np.hstack((X,X10)) # Padding array of zeros

Z2 = X

# Normalize input features to be between 0 and pi
for features in range(17):
  Z2[:,features] = 2*np.pi*(X[:,features] - np.min(X[:,features]))/(np.max(X[:,features])-np.min(X[:,features]))

# Normalize output features to be between 0 and 1 with some room on either side
Y_max = 15
Y_min = 1
Y = 2*(y - Y_min)/(Y_max-Y_min)-1

# Save the sets of features used in training
x_input = Z2
y_input = Y

In [8]:
# from Reference [2]
@qp.qnode(dev,interface="jax") # Initialize the qubit to interface with jax
def quantum_neural_network(params, x):
    layers=len(params[:,0,0])-1 # Input number of layers
    n_wires=len(params[0,:,0])  # number of wires
    n_params_rot=len(params[0,0,:]) # number of rotation parameters

    for j in range(N):
      qp.Hadamard(j)

    W(params[0,:,:].reshape(1,n_wires,n_params_rot))
    N_loop = int(np.ceil(17/N))
    N_set = N
    for i in range(layers):
      Set = i % N_loop

      start = N_set * Set
      end = start + N_set
      if N == 1:
        qp.RZ(x[start],0)
      else:
        SZ(x[start:end])
      W(params[i+1,:,:].reshape(1, n_wires, n_params_rot))

    observable = qp.PauliZ(0)

    for j in range(1, N):
      observable = observable @ qp.PauliZ(j)

    return qp.expval(observable)



In [9]:
# from Reference [2]

@jax.jit # uses jax's just in time (JIT) compiler
def mse(params,x,targets): # Define a mean square error function
    # We compute the mean square error between the target function and the quantum circuit to quantify the quality of our estimator
    return jnp.abs(quantum_neural_network(params,x)-jnp.array(targets))**2 #compare the value found using the QNN to the output feature (target)
@jax.jit # uses jax's just in time (JIT) compiler
def loss_fn(params, x,targets): # define a loss function
    # We define the loss function to feed our optimizer
    mse_pred = jax.vmap(mse,in_axes=(None, 0,0))(params,x,targets) # Calculate mse, in_axes specifies how multiplication is performed
    loss = jnp.mean(mse_pred) # define loss as the mean mse
    return loss



def Jax_Test_Train_Split(X, y, test_size):

    samples = X.shape[0]
    test_samples = int(round(samples * test_size))
    indices = np.random.permutation(samples)

    test_indices = indices[:test_samples]
    train_indices = indices[test_samples:]

    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]

    return X_train, X_test, y_train, y_test, test_indices



Schedule = optax.schedules.exponential_decay(0.25, 500, 0.9, staircase=False)
opt = optax.adamw(learning_rate=Schedule) # Set the optimizing routine and its learning rate
max_steps = 5000 # Set the number of max steps


@jax.jit # uses jax's just in time (JIT) compiler
def update_step_jit(i, args): # function to update the parameters

    params, opt_state, data, targets, print_training, best_params, best_loss, loss_array = args # unbundle the input arguments

    # Calculate loss and gradients for the adam optimization
    loss_val, grads = jax.value_and_grad(loss_fn)(params, data, targets)

    # Adam update using update and apply_updates
    updates, opt_state = opt.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    # Calculate loss of the updated parameters
    new_loss = loss_fn(params, data, targets)
    loss_array = loss_array.at[0,i].set(new_loss)
    # Update best parameters if new loss is lower
    def update_best():
        return params, new_loss

    def keep_best():
        return best_params, best_loss
    # Determine if the new parameters need to be saved using a coparison between the new and old loss
    best_params, best_loss = jax.lax.cond(new_loss < best_loss,update_best,keep_best)

    # Print every 10 steps if requested
    def print_fn():
        jax.debug.print("Step: {i}  Loss: {loss}", i=i,loss=new_loss)

    # Determine if the output should be printed
    jax.lax.cond((jnp.mod(i, 500) == 0) & print_training,print_fn,lambda: None)

    # return the original arguments
    return (params,opt_state,data,targets,print_training,best_params,best_loss,loss_array)


@jax.jit # uses jax's just in time (JIT) compiler
def optimization_jit(params, data, targets, print_training=False):

    # Initial best values
    best_loss = jnp.inf # infinity
    best_params = params # random values
    loss_array = np.zeros([1,max_steps])
    # Initialize the optimizer
    opt_state = opt.init(params)

    # Bundle a list of variables as args
    args = (params, opt_state, jnp.asarray(data), targets, print_training, best_params, best_loss, loss_array)

    # Optimization loop
    (params, opt_state, _, _, _, best_params, best_loss, loss_array) = jax.lax.fori_loop(0,max_steps + 1,update_step_jit,args)

    return best_params, best_loss, loss_array
# Evalutate the best set of parameters from training
def evaluate(params, data):
    return jax.vmap(quantum_neural_network,in_axes=(None, 0))(params, jnp.asarray(data))

In [10]:
# from Reference [2]
Iterations = 100 # Number of outer loops
miny1=1000
miny2=1000
miny3=1000
params_shape = qp.StronglyEntanglingLayers.shape(n_layers=layers+1,n_wires=wires) # Determine size of parameter array
miny = np.zeros([Iterations,3])
# Run training
for i in range(Iterations):
  print("Iteration: %f" % i)
  params = pnp.random.default_rng().random(size=params_shape) # Set random parameters

  X_train, X_test, y_train, y_test, indices = Jax_Test_Train_Split(x_input, y_input, Perc_Train)
  best_params, best_loss, loss_array = optimization_jit(params,X_train,jnp.array(y_train),print_training=True)


  y_predictions=evaluate(best_params,x_input)
  yn = (y_predictions+1)/2*(Y_max-Y_min)+Y_min
  miny[i,0] = max(100.*abs(yn-y)/y)
  miny[i,1] = np.mean(100.*abs(yn-y)/y)
  miny[i,2] = max(abs(np.log(yn/y)))
  if miny1>miny[i,0]:
    best_params_prime1 = best_params
    indices_prime1 = indices
    miny1=miny[i,0]
    print("Miny1: %f" % miny1)

  if miny2>miny[i,1]:
    best_params_prime2 = best_params
    indices_prime2 = indices
    miny2=miny[i,1]
    print("Miny2: %f" % miny2)

  if miny3>miny[i,2]:
    best_params_prime3 = best_params
    indices_prime3 = indices
    miny3=miny[i,2]
    print("Miny3: %f" % miny3)

Iteration: 0.000000
Step: 0  Loss: 0.869020931228979
Step: 500  Loss: 0.00014885865463199903
Step: 1000  Loss: 9.5267897327211e-05
Step: 1500  Loss: 0.0001650698172935648
Step: 2000  Loss: 5.518774947506806e-05
Step: 2500  Loss: 6.904892144947196e-05
Step: 3000  Loss: 4.4328436431880794e-05
Step: 3500  Loss: 2.868931257488485e-05
Step: 4000  Loss: 0.0002590153451093922
Step: 4500  Loss: 7.565506424822978e-05
Step: 5000  Loss: 5.0125061492470186e-05
Miny1: 189.277375
Miny2: 23.507075
Miny3: 1.062216
Iteration: 1.000000
Step: 0  Loss: 0.44676461281833724
Step: 500  Loss: 3.541197684612633e-08
Step: 1000  Loss: 0.0002650640732786284
Step: 1500  Loss: 4.8054089930491044e-05
Step: 2000  Loss: 0.0013153797639862447
Step: 2500  Loss: 3.737843192432647e-05
Step: 3000  Loss: 2.5060672776747225e-06
Step: 3500  Loss: 2.909366456010553e-07
Step: 4000  Loss: 0.0002471890109925911
Step: 4500  Loss: 5.274926224775246e-05
Step: 5000  Loss: 3.123761978382849e-06
Miny2: 20.672011
Iteration: 2.000000
Ste

In [11]:
#qp.draw_mpl(quantum_neural_network)(best_params_prime2,X_train[1,0:N])
print(qp.draw(quantum_neural_network, level="device")(best_params_prime2,X_train[1,:]))

0: ──H──Rot(0.01,1.30,-0.37)─╭●─╭X──RZ(1.46)──Rot(-0.33,0.43,0.10)─╭●─╭X──RZ(0.36) ···
1: ──H──Rot(-0.08,1.50,0.05)─╰X─╰●──RZ(3.16)──Rot(0.45,0.57,0.29)──╰X─╰●──RZ(5.09) ···

0: ··· ──Rot(1.04,0.23,-1.53)──╭●─╭X──RZ(4.62)──Rot(1.17,0.80,-1.56)──╭●─╭X──RZ(3.81) ···
1: ··· ──Rot(-0.97,-0.55,1.26)─╰X─╰●──RZ(4.35)──Rot(-0.05,1.17,-0.10)─╰X─╰●──RZ(1.02) ···

0: ··· ──Rot(0.10,1.01,0.09)──╭●─╭X──RZ(5.09)──Rot(-1.17,-2.12,-0.10)─╭●─╭X──RZ(4.20) ···
1: ··· ──Rot(0.87,0.29,-1.10)─╰X─╰●──RZ(5.84)──Rot(0.96,0.68,2.03)────╰X─╰●──RZ(5.90) ···

0: ··· ──Rot(1.90,0.46,0.76)───╭●─╭X──RZ(2.65)──Rot(-0.83,0.35,4.64)─╭●─╭X──RZ(3.59) ···
1: ··· ──Rot(-0.36,0.25,-0.95)─╰X─╰●──RZ(2.23)──Rot(0.89,0.09,1.82)──╰X─╰●──RZ(3.05) ···

0: ··· ──Rot(2.21,1.51,1.04)───╭●─╭X──RZ(0.85)──Rot(3.21,-0.29,0.74)─╭●─╭X─┤ ╭<Z@Z>
1: ··· ──Rot(-0.46,-0.56,2.52)─╰X─╰●──RZ(0.00)──Rot(0.41,0.02,0.15)──╰X─╰●─┤ ╰<Z@Z>


In [12]:
# From Reference [2]
print(miny1) # Print the best loss from training
print(miny2) # Print the best loss from training
print(miny3) # Print the best loss from training
y_predictions=evaluate(best_params_prime2,x_input) # Evalutate the results of training
print("Max predicted value:", max(y_predictions))
print("Min predicted value:", min(y_predictions))

44.30477360791827
7.904870382866913
0.5148538965673141
Max predicted value: 0.943705832233062
Min predicted value: -0.926538048613657


In [13]:
# From Reference [1]
yn = (y_predictions+1)/2*(Y_max-Y_min)+Y_min # reverse the normalization on the output parameters

print("Qubits:",N , "Test Samples:",Perc_Train , "Parameters:",(layers+1)*3*N,"Layers",layers, "Mean Error:", np.mean(100.*abs(yn-y)/(y)), "Max Error:", max(100.*abs(yn-y)/(y)))
# Print
print("PREDICTED k_s/R_a= ")
print(yn)
#print("Max err: %f" % max(100.*abs(yn-y)/(y)))
#print("mean err: %f" % np.mean(100.*abs(yn-y)/(y)))
print(np.sort(indices_prime2))

# Print the errors in a table format
Error=pd.DataFrame()
Error["$k_s/Ra$"]= y
Error["$k_{sp}/Ra$"]= yn
Error["$error$"]= (100.*(yn-y)/(y))

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(Error)

Qubits: 2 Test Samples: 0.3 Parameters: 60 Layers 9 Mean Error: 7.904870382866913 Max Error: 67.34917796929335
PREDICTED k_s/R_a= 
[ 3.32326816  4.17237426  4.01212516  9.28498017 12.68589305  7.59249306
 10.62371437 14.60594083 10.20697643  7.1343356  10.98824204  2.93383127
  9.46359145 10.84400879  9.52497967  7.59338432  8.55384079  7.5165988
  8.66773404 10.89960948  5.16261508  3.76057163  6.49100436  3.14971468
  3.78868796  7.17588874  3.89903595  6.78721471  1.96762501  7.68931209
  3.24968014  3.04344346  6.8838669   4.41084829  3.08616049  3.58099305
  2.33862478  1.53045387  1.86072159  2.85053505  2.27688262  2.55158291
  3.42483519  6.29828688  1.51423366]
[ 2  3  5 13 17 19 22 23 24 25 28 29 32 44]
     $k_s/Ra$  $k_{sp}/Ra$    $error$
0    3.319588     3.323268   0.110873
1    4.174497     4.172374  -0.050842
2    3.051813     4.012125  31.466919
3    7.168421     9.284980  29.526155
4   12.688976    12.685893  -0.024299
5    7.243094     7.592493   4.823893
6   10.6284